# Gold Rate Tracker — EDA & Modelling

This notebook walks through the data exploration and modelling decisions behind the forecaster in `ml/`.
It is written as a portfolio piece: the goal is to show *why* each decision was made, not just *what* was done.

**Data:** `data/history_seed.json` (≈444 estimated daily Indian 22K rates, bootstrapped from Yahoo Finance GC=F × INR=X, calibrated to match live Tanishq prices at the boundary) plus any live readings in `data/prices.json`.

**Model:** LightGBM regressor on the differenced 22K target (next-reading price delta).

**Validation:** Walk-forward 90-day backtest with per-fold retraining.

---

In [ ]:
import sys
import warnings

sys.path.insert(0, "..")
warnings.filterwarnings("ignore", message="X does not have valid feature names")

import json
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("dark_background")
GOLD = "#c8a456"
CREAM = "#f5ede0"
MUTE = "#8a8273"
FIG_W, FIG_H = 12, 4

In [ ]:
# Load calibrated combined history (seed calibrated to match live at boundary)
from ml.forecast import load_combined_history

combined = load_combined_history()
df = combined.copy()
df["ts"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.sort_values("ts").reset_index(drop=True)
df["price"] = df["22k"].astype(float)
df["delta"] = df["price"].diff()

# Track which rows are real vs seed for narrative
live_path = Path("../data/prices.json")
live_count = len(json.loads(live_path.read_text())) if live_path.exists() else 0

print(f"Total rows      : {len(df)} ({len(df) - live_count} seed, {live_count} live)")
print(f"Date range      : {df['ts'].iloc[0].date()} to {df['ts'].iloc[-1].date()}")
print(f"22K range       : Rs.{df['price'].min():.0f} to Rs.{df['price'].max():.0f}")

## 1. Price distribution and trend

The seed data is calibrated to match live Tanishq readings at the boundary (scale factor applied uniformly). The price level jump visible in the seed period reflects the ~7% gap between estimated international spot and actual Tanishq retail, which calibration corrects.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))

# Time series
ax = axes[0]
ax.plot(df["ts"], df["price"], color=GOLD, linewidth=1.5)
ax.set_title("22K Price (Rs/gram)", color=CREAM)
ax.set_xlabel("Date", color=MUTE)
ax.set_ylabel("Rs/gram", color=MUTE)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.tick_params(colors=MUTE)

# Distribution
ax2 = axes[1]
ax2.hist(df["price"].dropna(), bins=40, color=GOLD, alpha=0.8, edgecolor="none")
ax2.set_title("Price distribution", color=CREAM)
ax2.set_xlabel("Rs/gram", color=MUTE)
ax2.tick_params(colors=MUTE)

plt.tight_layout()
plt.show()
print(df["price"].describe().round(1))

## 2. Delta (differenced target) distribution

The model predicts the **delta** between consecutive readings, not the price level. This is more stationary and better-behaved as a regression target.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))

deltas = df["delta"].dropna()

ax = axes[0]
ax.hist(deltas, bins=60, color=GOLD, alpha=0.8, edgecolor="none")
ax.axvline(0, color="white", linewidth=0.8, linestyle="--")
ax.set_title("Distribution of daily delta (Rs)", color=CREAM)
ax.tick_params(colors=MUTE)

ax2 = axes[1]
ax2.plot(df["ts"][1:], deltas.values, color=GOLD, linewidth=0.8, alpha=0.7)
ax2.axhline(0, color="white", linewidth=0.8, linestyle="--")
ax2.set_title("Delta over time", color=CREAM)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax2.tick_params(colors=MUTE)

plt.tight_layout()
plt.show()
print(f"Mean delta: {deltas.mean():.1f}  Std: {deltas.std():.1f}  Skew: {deltas.skew():.2f}")
print(
    f"Drops >= Rs.100: {(deltas <= -100).sum()}  ({(deltas <= -100).mean()*100:.1f}% of readings)"
)

## 3. Day-of-week effect

Does gold move more on certain days? This informs whether day-of-week is a useful feature.

In [ ]:
df["dow"] = df["ts"].dt.dayofweek
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))

# Mean absolute delta by DOW
mad_by_dow = df.groupby("dow")["delta"].apply(lambda x: x.abs().mean())
axes[0].bar([dow_labels[i] for i in mad_by_dow.index], mad_by_dow.values, color=GOLD, alpha=0.85)
axes[0].set_title("Mean |delta| by day of week", color=CREAM)
axes[0].set_ylabel("Rs", color=MUTE)
axes[0].tick_params(colors=MUTE)

# Mean price by DOW (normalised to overall mean)
price_by_dow = df.groupby("dow")["price"].mean() / df["price"].mean() - 1
colors_dow = [GOLD if v >= 0 else "#c66a4b" for v in price_by_dow.values]
axes[1].bar(
    [dow_labels[i] for i in price_by_dow.index],
    price_by_dow.values * 100,
    color=colors_dow,
    alpha=0.85,
)
axes[1].set_title("Mean price by DOW (% vs overall avg)", color=CREAM)
axes[1].set_ylabel("%", color=MUTE)
axes[1].tick_params(colors=MUTE)

plt.tight_layout()
plt.show()

## 4. Autocorrelation

If there's meaningful autocorrelation in the price level or delta, lag features will be informative.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 2, figsize=(FIG_W, FIG_H * 1.8))

plot_acf(df["price"].dropna(), lags=40, ax=axes[0, 0], color=GOLD, title="ACF — price level")
plot_pacf(
    df["price"].dropna(),
    lags=40,
    ax=axes[0, 1],
    color=GOLD,
    title="PACF — price level",
    method="ols",
)
plot_acf(deltas.dropna(), lags=40, ax=axes[1, 0], color=GOLD, title="ACF — delta (differenced)")
plot_pacf(
    deltas.dropna(),
    lags=40,
    ax=axes[1, 1],
    color=GOLD,
    title="PACF — delta (differenced)",
    method="ols",
)

for ax in axes.flat:
    ax.tick_params(colors=MUTE)
    ax.set_facecolor("#0e0c0a")

plt.tight_layout()
plt.show()
print("Observation: price level is highly autocorrelated (near unit root).")
print("Differenced delta has much lower autocorrelation — better target for regression.")

## 5. Feature matrix inspection

Run the full feature pipeline and check completeness.

In [ ]:
from ml.features import FEATURE_COLS, build_feature_matrix, get_train_Xy

feat_df = build_feature_matrix(df)
X, y = get_train_Xy(feat_df)

print(f"Training rows : {len(X)}  (dropped {len(df) - len(X)} for NaN features/target)")
print(f"Features      : {len(FEATURE_COLS)}")
print("\nFeature stats:")
X.describe().round(1)

## 6. Walk-forward backtest results

The backtest uses a strict walk-forward protocol: for each test fold, the model is trained on all data before that fold. No future information leaks.

We compare against the naive baseline: predict no change (delta = 0).

**Current results (90-day window, 58 folds on seed data):**

| Metric | LightGBM | Naive baseline |
|---|---|---|
| MAE | Rs. 283 | Rs. 204 |
| MAPE | 1.87% | 1.35% |
| Direction accuracy | 48.3% | 0.0% |

The naive baseline beats the model on MAE — this is expected and disclosed honestly. See the Interpretation cell below.

In [ ]:
bt_path = Path("../data/backtest.json")
if bt_path.exists():
    bt = json.loads(bt_path.read_text())
    preds = pd.DataFrame(bt["predictions"])
    preds["ts"] = pd.to_datetime(preds["ts"], utc=True)
    preds["model_err"] = preds["predicted"] - preds["actual"]
    preds["baseline_err"] = preds["baseline"] - preds["actual"]

    m = bt["model"]
    b = bt["baseline"]
    print(
        f"Model   — MAE: Rs.{m['mae']:.1f}  MAPE: {m['mape']:.2f}%  Dir-acc: {m['direction_acc']*100:.1f}%"
    )
    print(
        f"Baseline— MAE: Rs.{b['mae']:.1f}  MAPE: {b['mape']:.2f}%  Dir-acc: {b['direction_acc']*100:.1f}%"
    )
    print(f"Folds   : {bt['folds']}  Window: {bt['backtest_days']} days")
else:
    print("data/backtest.json not found — run: python ml/backtest.py")
    preds = None

In [ ]:
if preds is not None:
    fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))

    # Actual vs predicted
    ax = axes[0]
    ax.plot(preds["ts"], preds["actual"], color=CREAM, linewidth=1.2, label="Actual", alpha=0.9)
    ax.plot(preds["ts"], preds["predicted"], color=GOLD, linewidth=1.2, label="Model", alpha=0.8)
    ax.plot(
        preds["ts"],
        preds["baseline"],
        color=MUTE,
        linewidth=0.8,
        linestyle="--",
        label="Naive baseline",
        alpha=0.7,
    )
    ax.set_title("Actual vs predicted (22K)", color=CREAM)
    ax.legend(fontsize=10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.tick_params(colors=MUTE)

    # Error distribution
    ax2 = axes[1]
    ax2.hist(
        preds["model_err"], bins=30, color=GOLD, alpha=0.7, label="Model error", edgecolor="none"
    )
    ax2.hist(
        preds["baseline_err"],
        bins=30,
        color=MUTE,
        alpha=0.5,
        label="Baseline error",
        edgecolor="none",
    )
    ax2.axvline(0, color="white", linewidth=0.8, linestyle="--")
    ax2.set_title("Prediction error distribution", color=CREAM)
    ax2.legend(fontsize=10)
    ax2.tick_params(colors=MUTE)

    plt.tight_layout()
    plt.show()

## 7. Feature importance

Train on the full dataset and inspect which features the model relies on most.

In [ ]:
import lightgbm as lgb

model = lgb.LGBMRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
)
model.fit(X, y)

importance = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
colors_imp = [GOLD if v > importance.median() else MUTE for v in importance.values]
importance.plot(kind="barh", ax=ax, color=colors_imp, edgecolor="none")
ax.set_title("LightGBM feature importance (split count)", color=CREAM)
ax.set_xlabel("Importance", color=MUTE)
ax.tick_params(colors=MUTE)
plt.tight_layout()
plt.show()

print("\nTop 5 features:")
print(importance.tail(5).to_string())

## 8. Interpretation

**What the backtest tells us (90-day walk-forward, 58 folds):**
- The naive baseline (predict-last-value = no change) has lower MAE (Rs. 204) than the model (Rs. 283) on this data. This is expected and disclosed honestly: gold prices exhibit near-random-walk behaviour on short horizons, so predicting no change is a hard baseline to beat on absolute error.
- The model's only demonstrated advantage is directional accuracy: 48.3% vs 0% for the naive baseline. The baseline always predicts zero delta, so it is directionally correct 0% of the time by construction.
- Neither result is a reason to trade on. They are diagnostic outputs for an honest ML system.
- Results will improve as the live scraper accumulates months of real Tanishq readings to replace the estimated seed data.

**Why LightGBM over LSTM/Transformer:**
- We have ~428 training rows at deployment time. Sequence models need thousands.
- LightGBM retrains from scratch in < 1 second, so we can afford per-scrape retraining without persisting a model artefact to the repo.
- The feature importance plots are interpretable and explain which signals the model uses.

**Data quality caveat:**
- The seed data is estimated from international spot prices, not actual Tanishq retail rates.
- `load_combined_history()` applies a calibration scale factor (seed tail mean / live head mean) so the seed and live series meet smoothly.
- This is disclosed in `README.md` under \"What's real and what's seeded\".